# 05: Source Quality Assessment

Not all polities in CrisisDB have equal documentary evidence. Ancient polities often have fragmentary records where absence of documented conflict may reflect gaps in sources rather than actual peaceful transitions.

This notebook flags polities by source quality based on:
1. Time period (older = sparser records)
2. Region and documentary tradition
3. Expert feedback (Jenny Reddish, Peter Turchin, CSH Vienna)

**Key feedback from CSH team (Jan 2026):**
- Aztec Empire: "we don't have the same detail in the sources on precolonial Aztec power transmission"
- Egyptian Old Kingdom: "lacks rich documentary sources about ruler transitions"
- Venice, Northern Song: valid outliers with good institutional records

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('power_transitions.csv', sep='|')
print(f"Total transitions: {len(df)}")
print(f"Total polities: {df['polity_name'].nunique()}")

Total transitions: 3447
Total polities: 263


## Source Quality Classification

Three tiers:
- **High**: Rich documentary tradition, multiple independent sources (post-1000 CE, well-documented empires)
- **Medium**: Reasonable records but some gaps (500 BCE - 1000 CE)
- **Low**: Sparse, fragmentary, or heavily reconstructed records (pre-500 BCE, specific problematic polities)

In [2]:
# Polities with known source quality issues (from CSH feedback)
LOW_QUALITY_POLITIES = [
    'Aztec Empire',
    'Egypt - Classic Old Kingdom',
    'Egypt - Late Old Kingdom',
    'Egypt - Period of the Regions',
    'Egypt - Middle Kingdom',
    'Egypt - New Kingdom Ramesside Period',
    'Egypt - New Kingdom Thutmosid Period',
    'Egypt - Thebes-Hyksos Period',
    'Early Classic Tikal',
    'Hephthalite Empire',
    'Rouran Khaganate',
    'Akkadian Empire',
    'Hatti - New Kingdom',
]

def classify_quality(row):
    if row['polity_name'] in LOW_QUALITY_POLITIES:
        return 'low'
    if row['transition_year'] < -1000:
        return 'low'
    elif row['transition_year'] < 1000:
        return 'medium'
    return 'high'

df['quality'] = df.apply(classify_quality, axis=1)
print("Quality distribution:")
print(df['quality'].value_counts())

Quality distribution:
high      2387
medium     926
low        134
Name: quality, dtype: int64


## Impact on Key Findings

Do our main results hold when excluding low-quality sources?

In [3]:
df_hq = df[df['quality'] != 'low'].copy()
print(f"After filtering: {len(df_hq)} transitions ({len(df_hq)/len(df)*100:.1f}%)")
print(f"Polities remaining: {df_hq['polity_name'].nunique()} (of {df['polity_name'].nunique()})")

After filtering: 3313 transitions (96.1%)
Polities remaining: 250 (of 263)


In [4]:
# Violence codes: P=present, IP=inferred present, SU=suspected
VIOLENT_CODES = ['P', 'IP', 'SU']

def is_violent(row):
    for col in ['intra_elite', 'military_revolt', 'popular_uprising', 'predecessor_assassination']:
        if row.get(col) in VIOLENT_CODES:
            return True
    return False

# Full dataset
df['violent'] = df.apply(is_violent, axis=1)
df_sorted = df.sort_values(['polity_name', 'transition_year'])
df_sorted['prev_violent'] = df_sorted.groupby('polity_name')['violent'].shift(1)
df_pairs_full = df_sorted.dropna(subset=['prev_violent'])

vv_full = df_pairs_full[(df_pairs_full['prev_violent'] == True) & (df_pairs_full['violent'] == True)].shape[0]
vp_full = df_pairs_full[(df_pairs_full['prev_violent'] == True) & (df_pairs_full['violent'] == False)].shape[0]
pv_full = df_pairs_full[(df_pairs_full['prev_violent'] == False) & (df_pairs_full['violent'] == True)].shape[0]
pp_full = df_pairs_full[(df_pairs_full['prev_violent'] == False) & (df_pairs_full['violent'] == False)].shape[0]

p_vv_full = vv_full / (vv_full + vp_full)
p_pv_full = pv_full / (pv_full + pp_full)

# Filtered dataset
df_hq['violent'] = df_hq.apply(is_violent, axis=1)
df_hq_sorted = df_hq.sort_values(['polity_name', 'transition_year'])
df_hq_sorted['prev_violent'] = df_hq_sorted.groupby('polity_name')['violent'].shift(1)
df_pairs_hq = df_hq_sorted.dropna(subset=['prev_violent'])

vv_hq = df_pairs_hq[(df_pairs_hq['prev_violent'] == True) & (df_pairs_hq['violent'] == True)].shape[0]
vp_hq = df_pairs_hq[(df_pairs_hq['prev_violent'] == True) & (df_pairs_hq['violent'] == False)].shape[0]
pv_hq = df_pairs_hq[(df_pairs_hq['prev_violent'] == False) & (df_pairs_hq['violent'] == True)].shape[0]
pp_hq = df_pairs_hq[(df_pairs_hq['prev_violent'] == False) & (df_pairs_hq['violent'] == False)].shape[0]

p_vv_hq = vv_hq / (vv_hq + vp_hq)
p_pv_hq = pv_hq / (pv_hq + pp_hq)

print("=== COMPARISON: Full vs Filtered ===\n")
print(f"{'Metric':<35} {'Full':>12} {'Filtered':>12}")
print("-" * 60)
print(f"{'Transitions':<35} {len(df):>12} {len(df_hq):>12}")
print(f"{'Polities':<35} {df['polity_name'].nunique():>12} {df_hq['polity_name'].nunique():>12}")
print(f"{'P(violent|prev violent)':<35} {p_vv_full:>11.1%} {p_vv_hq:>11.1%}")
print(f"{'P(violent|prev peaceful)':<35} {p_pv_full:>11.1%} {p_pv_hq:>11.1%}")
print(f"{'Contagion ratio':<35} {p_vv_full/p_pv_full:>11.2f}x {p_vv_hq/p_pv_hq:>11.2f}x")

=== COMPARISON: Full vs Filtered ===

Metric                                      Full     Filtered
------------------------------------------------------------
Transitions                                 3447         3313
Polities                                     263          250
P(violent|prev violent)                    72.8%        73.0%
P(violent|prev peaceful)                   28.7%        28.2%
Contagion ratio                            2.54x        2.59x


## Low Quality Polities

These 13 polities are flagged for sparse documentary records.

In [5]:
low_q = df[df['quality'] == 'low'].groupby('polity_name')['transition_year'].min().reset_index()
low_q = low_q.sort_values('polity_name')

print(f"Low quality polities ({len(low_q)}):")
for _, r in low_q.iterrows():
    print(f"  - {r['polity_name']} ({int(r['transition_year'])})")

Low quality polities (13):
  - Akkadian Empire (-2279)
  - Aztec Empire (1440)
  - Early Classic Tikal (292)
  - Egypt - Classic Old Kingdom (-2667)
  - Egypt - Late Old Kingdom (-2323)
  - Egypt - Middle Kingdom (-1992)
  - Egypt - New Kingdom Ramesside Period (-1294)
  - Egypt - New Kingdom Thutmosid Period (-1525)
  - Egypt - Period of the Regions (-2125)
  - Egypt - Thebes-Hyksos Period (-1555)
  - Hatti - New Kingdom (-1322)
  - Hephthalite Empire (460)
  - Rouran Khaganate (410)


## Summary

**Result**: Filtering low-quality sources does not change core findings. The violence contagion ratio actually increases slightly from 2.54x to 2.59x.

| Finding | Full Dataset | Filtered |
|---------|-------------|----------|
| P(violent \| prev violent) | 72.8% | 73.0% |
| P(violent \| prev peaceful) | 28.7% | 28.2% |
| Violence contagion ratio | 2.54x | 2.59x |

The web UI includes a toggle to exclude low-quality polities, allowing users to verify robustness.